In [1]:
import numpy as np
import matplotlib.pyplot as plt
import re
import json
from functions import *
import pandas as pd
import os
import time
import threading
from http.server import SimpleHTTPRequestHandler
from socketserver import TCPServer
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pickle

def get_currents(file):
    def read_currents_file(filepath):
        # Read the file content
        with open(filepath, 'r') as file:
            # Read the first line to get current names
            header_line = file.readline().strip()
            
            # Split the header first by semicolons, then by commas
            current_groups = header_line.split(';')
            current_names = []
            for group in current_groups:
                currents = [name.strip() for name in group.split(',')]
                current_names.extend(currents)
                
            # Initialize dictionary with empty lists for each current
            data_dict = {name: [] for name in current_names}
            
            # Read the rest of the lines
            for line in file:
                # if start with undefined,skip
                if line.startswith("undefined"):
                    continue
                if not line.strip():  # Skip empty lines
                    continue
                
                # Split values by semicolon first, then comma
                value_groups = line.strip().split(';')
                values = []
                for group in value_groups:
                    group_values = [float(val.strip()) for val in group.split(',') if val.strip()]
                    values.extend(group_values)
                
                # Add each value to corresponding current's list
                for name, value in zip(current_names, values):
                    data_dict[name].append(value)
        
        # Convert lists to numpy arrays for easier manipulation
        for name in data_dict:
            data_dict[name] = np.array(data_dict[name])
        
        return data_dict

    # Example usage:
    filepath = file
    currents_data = read_currents_file(filepath)
    return currents_data
def readFile(file,unidentifiable_space = [3,5,6,7,8,9,10,11]):
    if not unidentifiable_space:
        unidentifiable_space = list(range(12))
    currents_data = get_currents(file)
    mask =  -100< currents_data['voltage']
    if 'NA' in currents_data:
        del currents_data['NA']
    matrix = np.zeros((len(currents_data['voltage'][mask]),len(currents_data)-1))  # Exclude voltage
    # assign currents to matrix columns
    for i, (current_name, values) in enumerate(currents_data.items()):
        if current_name not in ['voltage']:
            matrix[:, i - (1 if 'voltage' in currents_data else 0)] = values[mask]
    U, S, V = np.linalg.svd(matrix)
    def projection_S(v):
        ps = [0] * len(S)
        for i in unidentifiable_space:
            ps += np.inner(v,V[i]) / np.inner(V[i],V[i]) * V[i]
        return ps
    identifiability = {}
    for i, (current_name, values) in enumerate(currents_data.items()):
        if current_name in ['voltage']:
            continue
        v_I = [0]*len(S)
        v_I[i-1] = 1
        k = np.linalg.norm(v_I-projection_S(v_I))
        identifiability[current_name] = k
    sorted_identifiability = dict(sorted(identifiability.items(), key=lambda item: item[1], reverse=True))
    return U,S,V,currents_data,sorted_identifiability
def plot_sorted_currents_with_identifiability(currents_data, sorted_identifiability):
    # Get currents in sorted order (already sorted by identifiability)
    currents_to_plot = list(sorted_identifiability.keys())
    n_currents = len(currents_to_plot)
    
    # Create figure with special grid
    fig = plt.figure(figsize=(15, 4*(n_currents//3 + 2)))  # +2 for voltage row
    
    # Create grid with different row heights
    gs = plt.GridSpec(n_currents//3 + 2, 3, height_ratios=[1.5] + [1]*(n_currents//3 + 1))
    
    # Plot voltage across entire first row
    ax_voltage = fig.add_subplot(gs[0, :])
    ax_voltage.plot(currents_data['voltage'], 'b-')
    ax_voltage.set_title('Voltage')
    ax_voltage.set_ylabel('mV')
    ax_voltage.set_xlabel('Time Step')
    ax_voltage.grid(True)
    
    # Plot currents in remaining grid
    for idx, current_name in enumerate(currents_to_plot):
        row = (idx // 3) + 1  # +1 because voltage took first row
        col = idx % 3
        ax = fig.add_subplot(gs[row, col])
        
        # Plot the current
        ax.plot(currents_data[current_name], 'b-')
        
        # Set title with identifiability value
        identifiability_value = sorted_identifiability[current_name]
        def format_current_name(name):
            # Dictionary for special current name formatting
            current_formats = {
                'INa': 'I_{Na}',
                'ICaL': 'I_{CaL}',
                'Ito': 'I_{to}',
                'IKr': 'I_{Kr}',
                'IKs': 'I_{Ks}',
                'IK1': 'I_{K1}',
                'INaCa': 'I_{NaCa}',
                'INaK': 'I_{NaK}',
                'INab': 'I_{Nab}',
                'ICab': 'I_{Cab}',
                'IKb': 'I_{Kb}',
                'IpCa': 'I_{pCa}',
                'INalate': 'I_{Na,late}'
            }
            return current_formats.get(name, name)  # Return formatted name or original if not in dictionary

        # Then modify the title setting line to:
        ax.set_title(f'${format_current_name(current_name)}$\nIdentifiability: {identifiability_value:.3f}',fontsize=20)        
        ax.set_ylabel('Current (pA/pF)')
        ax.set_xlabel('Time Step')
        ax.grid(True)
    
    #plt.tight_layout()
    #plt.savefig('currents_sorted_by_identifiability.png', dpi=300)
    #plt.show()
    plt.close()
def run_simulation(file,pacing_period,drug_dict,drug_name):
    U,S,V,currents_data, sorted_identifiability = readFile(file)
    #print(S,'S',V,sorted_identifiability)
    #plot_sorted_currents_with_identifiability(currents_data, sorted_identifiability)

    ######################## generate the perturbed currents json file for simulation
    json_file_name = f'perturbed_currents_pacingperiod_{pacing_period}.js'
    json_file = f'./2D-TNNP-sensitivity-test/{json_file_name}'

    epsilon_lst = [-0.5,-0.2,0.,0.2,0.5]
    # now we perturb current by singular vector with magnitude epsilon
    # the order of current should be from dict current_data keys except voltage
    # now we store each singular vector's perturbation into a list as well
    perturbed_currents = {}
    for idx,singular_vector in enumerate(V):
        perturbed_currents[idx] = {}
        for epsilon in epsilon_lst:
            perturbed_currents[idx][epsilon] =[ 1.+ epsilon *  v for v in singular_vector]
    perturbed_currents_name = ['C_Na', 'C_to', 'C_CaL', 'C_Ks', 'C_pK', 'C_NaK', 'C_Kr', 'C_NaCa', 'C_K1', 'C_bCa', 'C_pCa', 'C_bNa']
    # now output the dict to a js file
    with open(json_file, 'w') as f:
        # 1. Write the names array
        
        f.write(f"const pacePeriod = {pacing_period};\n")
        f.write(f"const perturbed_currents_name = {json.dumps(perturbed_currents_name)};\n")
        f.write(f'const drug_name = "{drug_name}";\n')
        # 2. Write the dictionary (the data)
        # indent=4 makes it readable; without it, it stays on one line
        f.write(f"const perturbed_currents = {json.dumps(perturbed_currents, indent=4)};")

    ###### also copy .\2D-TNNP-sensitivity-test\drug_data.js to .\2D-TNNP-sensitivity-test\drug_data.js
    if drug_name not in drug_dict:
        print(f"Drug '{drug_name}' not found in the dataset.")
        return
    
    drug_data = drug_dict[drug_name]
    drug_data['drug_name'] = drug_name

    all_currents = ['INa', 'IKr', 'ICaL', 'INaL', 'IKs', 'Ito', 'IK1']
    for current in all_currents:
        if current not in drug_data:
            drug_data[current] = {
                "IC50": 0.0,
                "h": 1.0
            }

    output_filename = 'drug_data.js'
    output_path = './2D-TNNP-sensitivity-test'

    with open(f"{output_path}/{output_filename}", 'w') as js_file:
        js_file.write("const drugData = ")
        json.dump(drug_data, js_file, indent=4)
        js_file.write(";")  # End the JS variable declaration

    idx_file = './2D-TNNP-sensitivity-test/index.html'



    ########################## modify the simulation index file
    simulation_index_file = './2D-TNNP-sensitivity-test/index.html'
    indicator = "<script src='Abubu/libs/Abubu.js'></script>"
    target_pattern = r'<script src=".*?"></script>'

    # 3. Read the HTML file
    with open(simulation_index_file, 'r') as file:
        content = file.read()

    # 4. Split the content into two parts: before the indicator and after it
    if indicator in content:
        parts = content.split(indicator, 1) # Split only once
        header = parts[0] + indicator
        rest_of_file = parts[1]
        
        # 5. Replace only the FIRST occurrence of a script tag in the remaining text
        new_script_tag = f'<script src="{json_file_name}"></script>'
        updated_rest = re.sub(target_pattern, new_script_tag, rest_of_file, count=1)
        
        # 6. Reconstruct the full HTML
        final_html = header + updated_rest

        # 7. Write it back to the file
        with open(idx_file, 'r') as file:
            html_content = file.read()
            script_tag = f"<script src='{output_filename}'></script>"
            if script_tag not in html_content:
                insertion_point = html_content.find("<script src='Abubu/libs/Abubu.js'></script>") + len("<script src='Abubu/libs/Abubu.js'></script>")
                new_html_content = html_content[:insertion_point] + f"\n\n<script src='{output_filename}'></script>\n<script src='{json_file_name}'></script>" + html_content[insertion_point:]
                with open(idx_file, 'w') as file:
                    file.write(new_html_content)
        
        print(f"Successfully updated the line following {indicator}")
    else:
        print("Indicator line not found. No changes made.")



    PORT = 8001
    DIRECTORY = "2D-TNNP-sensitivity-test" # The folder containing your index.html
    TARGET_MESSAGE = "All simulations are done!"
    URL = f"http://localhost:{PORT}/index.html"

    def start_server():
        """Starts a local server in the specified directory."""
        os.chdir(os.path.abspath(DIRECTORY))
        # Allow restarting the script immediately without "Address already in use" errors
        TCPServer.allow_reuse_address = True
        with TCPServer(("", PORT), SimpleHTTPRequestHandler) as httpd:
            print(f"Serving at {URL}")
            httpd.serve_forever()
        
    # 1. Start the server in a background thread so the script can keep moving
    server_thread = threading.Thread(target=start_server, daemon=True)
    server_thread.start()

    # 2. Configure Chrome
    options = webdriver.ChromeOptions()
    prefs = {"profile.default_content_setting_values.automatic_downloads": 1}
    options.add_experimental_option("prefs", prefs)    
    options.set_capability('goog:loggingPrefs', {'browser': 'ALL'})
    # Optional: This keeps the driver logs quiet in your terminal
    options.add_experimental_option('excludeSwitches', ['enable-logging'])
    options.add_argument("--window-position=-500,1350")
    driver = webdriver.Chrome(options=options)

    try:
        # 3. Open the localhost URL
        driver.get(URL)
        print("Simulation started on localhost. Monitoring console...")
    
        # wait for 10 s
        time.sleep(10)
        # --- NEW: Automatically click the Solve/Pause button ---
        try:
            # 1. Look for the span containing 'Solve/Pause'
            # We use '*' because dat.GUI doesn't use standard <button> tags
            xpath_selector = "//*[contains(text(), 'Solve/Pause')]"
            
            # 2. Wait for the element to be present and visible
            solve_element = WebDriverWait(driver, 2).until(
                EC.visibility_of_element_located((By.XPATH, xpath_selector))
            )
            
            # 3. Click the element directly via Selenium
            solve_element.click()
            print("Clicked 'Solve/Pause' GUI element successfully.")
        except Exception as e:
            print(f"Could not find or click the button automatically: {e}")
        # -------------------------------------------------------
        running = True
        while running:
            logs = driver.get_log('browser')
            for entry in logs:
                # entry['message'] often contains extra info, so we check if our string is IN it
                if TARGET_MESSAGE.lower() in entry['message'].lower():
                    print(f"Match found: '{TARGET_MESSAGE}'. Finalizing...")
                    time.sleep(5) # Give you a moment to see the final state
                    running = False
                    break
            time.sleep(1)

    finally:
        print("Shutting down...")
        driver.quit()
        # The server thread will die automatically because it's a 'daemon'

In [2]:
cur_dir = os.getcwd()
need_to_redo = ['test1(cisapride)','test2(verapamil)','Amiodarone II',
 'Bepridil II',
 'Bepridil III',
 'Chloropromazine II',
 'Cisapride II',
 'Diltiazem II',
 'Dofetilide II',
 'Dofetilide III',
 'Flecainide II',
 'Flecainide III',
 'Lidocaine II',
 'Mexiletine II',
 'Mibefradil II',
 'Moxifloxacin II',
 'Moxifloxacin III',
 'Nilotinib II',
 'Quinidine',
 'Ranolazine',
 'Saquinavir',
 'Sertindole II',
 'Sotalol II',
 'Sparfloxacin II',
 'Terfenadine II',
 'Verapamil II',
 'Verapamil III']
need_to_redo = ['test1(cisapride)','test2(verapamil)']

In [3]:
os.chdir(cur_dir)
pacing_period = 1000
folder = f"2D-TNNP-pacing-period-{pacing_period}-5xdrug"

# check all csv files inside folder
csv_files = [f for f in os.listdir(folder) if f.endswith('.csv')]


# make csv files alphabet order
csv_files.sort()
# and find the pkl file
pkl_files = [f for f in os.listdir(folder) if f.endswith('.pkl')]

# create a txt file on Z://WillAn_Backup
with open("Z://WillAn_Backup/monitoring_simulations.txt", "w") as f:
    pass

for file in pkl_files:
    with open(os.path.join(folder, file), 'rb') as f:
        drug_dict = pickle.load(f)

for file in csv_files:
# file = "voltage_TNNP_pacingPeriod_{drug_name}.csv"
    os.chdir(cur_dir)

    drug_name = file.split('_')[-1].split('.')[0]  # Extract drug name from filename
    if 'INaL' in drug_dict[drug_name]:
        print(f"Skipping simulation for {drug_name} due to INaL involvement...")
        continue
    else:
        print(f"Running simulation for {drug_name}...")
        # and print the absolute path of the file to be simulated
        run_simulation(os.path.join(folder, file), pacing_period, drug_dict, drug_name)
        # once finish, write the drug name into the txt file with cur time
        with open("Z://WillAn_Backup/monitoring_simulations.txt", "a") as f:
            f.write(f"{drug_name} simulation completed at {time.strftime('%Y-%m-%d %H:%M:%S')}.\n")

Running simulation for Amiodarone I...
Successfully updated the line following <script src='Abubu/libs/Abubu.js'></script>
Serving at http://localhost:8001/index.html


127.0.0.1 - - [09/Apr/2026 16:00:00] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:00] "GET /libs/dat.gui.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:00] "GET /libs/stats.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:00] "GET /Abubu/libs/Abubu.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:00] "GET /drug_data.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:00] "GET /perturbed_currents_pacingperiod_1000.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:00] "GET /config.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:00] "GET /libs/require.js HTTP/1.1" 200 -


Simulation started on localhost. Monitoring console...


127.0.0.1 - - [09/Apr/2026 16:00:00] code 404, message File not found
127.0.0.1 - - [09/Apr/2026 16:00:00] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [09/Apr/2026 16:00:00] "GET /app/main.js?bust=1775764800642 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:01] "GET /libs/shader.js?bust=1775764800642 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:01] "GET /ComputeGL/ComputeGL.js?bust=1775764800642 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:01] "GET /libs/text.js?bust=1775764800642 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:01] "GET /app/shaders/vertShader.vert?bust=1775764800642&bust=1775764800642 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:01] "GET /app/shaders/initShader.frag?bust=1775764800642&bust=1775764800642 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:01] "GET /app/shaders/compShader.frag?bust=1775764800642&bust=1775764800642 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 16:00:01] "GET /app/shaders/getCurrentsShader.frag?bust=1775764800642&bust=1775764

Clicked 'Solve/Pause' GUI element successfully.
Match found: 'All simulations are done!'. Finalizing...
Shutting down...
Skipping simulation for Amiodarone II due to INaL involvement...
Running simulation for Astemizole...
Successfully updated the line following <script src='Abubu/libs/Abubu.js'></script>
Serving at http://localhost:8001/index.html


...

Clicked 'Solve/Pause' GUI element successfully.
Match found: 'All simulations are done!'. Finalizing...
Shutting down...
Running simulation for test3(none)...
Successfully updated the line following <script src='Abubu/libs/Abubu.js'></script>
Serving at http://localhost:8001/index.html


127.0.0.1 - - [10/Apr/2026 02:03:21] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 02:03:21] "GET /libs/dat.gui.js HTTP/1.1" 200 -


Simulation started on localhost. Monitoring console...


127.0.0.1 - - [10/Apr/2026 02:03:21] "GET /libs/stats.js HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 02:03:21] "GET /Abubu/libs/Abubu.js HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 02:03:21] "GET /drug_data.js HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 02:03:21] "GET /perturbed_currents_pacingperiod_1000.js HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 02:03:21] "GET /config.js HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 02:03:21] "GET /libs/require.js HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 02:03:21] code 404, message File not found
127.0.0.1 - - [10/Apr/2026 02:03:21] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [10/Apr/2026 02:03:21] "GET /app/main.js?bust=1775801001604 HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 02:03:22] "GET /libs/shader.js?bust=1775801001604 HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 02:03:22] "GET /ComputeGL/ComputeGL.js?bust=1775801001604 HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 02:03:22] "GET /libs/text.js?bust=1775801001604 HTTP/1.1" 200 -
127.0.0.1 - - [10

Clicked 'Solve/Pause' GUI element successfully.
Match found: 'All simulations are done!'. Finalizing...
Shutting down...
